Lab03
Luis Roberto Navarro Quinn

In [1]:
import findspark
findspark.init()

In [2]:
from spark_utils import SparkUtils

su = SparkUtils("Ejemplo Spark SQL", "spark://spark-master:7077")
su._spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 02:26:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=spark://spark-master:7077 appName=Ejemplo Spark SQL>

In [3]:
columns_types = [("Order_ID", "long"),
                 ("Company", "string"),
                 ("City", "string"),
                 ("Customer_Age", "int"),
                 ("Order_Value", "float"),
                 ("Delivery_Time_Min", "float"),
                 ("Distance_Km", "float"),
                 ("Items_Count", "float"),
                 ("Product_Category", "string"),
                 ("Payment_Method", "string"),
                 ("Customer_Rating", "float"),
                 ("Discount_Applied", "float"),
                 ("Delivery_Partner_Rating", "float")
                 ]


qcommerce_schema = SparkUtils.generate_schema(columns_types)

qcommerce_df = su._spark.read \
                    .option("header", "true") \
                    .schema(qcommerce_schema) \
                    .csv("/opt/spark/work-dir/data/qcommerce")

qcommerce_df.printSchema()
print(f'número de registros: {qcommerce_df.count()}')
qcommerce_df.show(5)

root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)



número de registros: 1000000
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.0|                    3

In [4]:
qcommerce_df.columns

['Order_ID',
 'Company',
 'City',
 'Customer_Age',
 'Order_Value',
 'Delivery_Time_Min',
 'Distance_Km',
 'Items_Count',
 'Product_Category',
 'Payment_Method',
 'Customer_Rating',
 'Discount_Applied',
 'Delivery_Partner_Rating']

In [5]:
from pyspark.sql.functions import when, count, isnull

null_count_df = qcommerce_df.select([count(when(isnull(c),c)).alias(c) for c in qcommerce_df.columns])
null_count_df.show()

[Stage 4:=============================>                             (1 + 1) / 2]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [6]:
null_count_df_2 = SparkUtils.count_null(qcommerce_df)
null_count_df_2.show()

[Stage 7:=============================>                             (1 + 1) / 2]

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [7]:
qcommerce_df_wo_nulls_dropna = qcommerce_df.dropna()
print(f'Número de registros despues de usar el metofo dropan(): {qcommerce_df_wo_nulls_dropna.count()}')

[Stage 10:=============================>                            (1 + 1) / 2]

Número de registros despues de usar el metofo dropan(): 780992


In [11]:
qcommerce_clean_df = qcommerce_df.fillna({
    'City': 'Uknown',
    'Items_Count': 0.0,
    'Customer_Rating': 0.0,
    'Delivery_Partner_Rating': 0.0
})
print(f"Número de registros despues de usar el metodo 'fillna()':{qcommerce_df_wo_nulls_fillna.count()}")

Número de registros despues de usar el metodo 'fillna()':1000000


In [12]:
from pyspark.sql.functions import lit
qcommerce_clean_df = qcommerce_clean_df.withColumn("Country_Code", lit("IN"))
qcommerce_clean_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Country_Code|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|          IN|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

In [19]:
from pyspark.sql.functions import col, when

qcommerce_task1_df = (qcommerce_clean_df.withColumn("Delivery_SLA", when(col("Delivery_Time_Min") <= 15, "FAST")
        .when((col("Delivery_Time_Min") > 15) & (col("Delivery_Time_Min") <= 20), "ON TIME")
        .when(col("Delivery_Time_Min") > 20, "LATE")
        .otherwise("UNKNOWN"))
        .orderBy(col("Delivery_Time_Min").desc())
        .select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA"))

qcommerce_task1_df.show()

[Stage 21:=============================>                            (1 + 1) / 2]

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        LATE|
| 1807126|Jio Mart|Haridwar|             40.0|        LATE|
| 1165764|Jio Mart|Haridwar|           39.994|        LATE|
| 1610720|Jio Mart|Haridwar|           39.994|        LATE|
| 1729503|Jio Mart|Haridwar|           39.994|        LATE|
| 1951122|Jio Mart|Haridwar|           39.988|        LATE|
| 1975896|Jio Mart|Haridwar|           39.988|        LATE|
| 1059671|Jio Mart|Haridwar|           39.982|        LATE|
| 1594835|Jio Mart|Haridwar|           39.982|        LATE|
| 1162804|Jio Mart|Haridwar|           39.982|        LATE|
| 1826295|Jio Mart|Haridwar|           39.982|        LATE|
| 1961544|Jio Mart|Haridwar|           39.976|        LATE|
| 1360875|Jio Mart|Haridwar|           39.964|        LATE|
| 1555058|Jio Mart|Haridwar|           3

In [38]:
from pyspark.sql.functions import avg, count

qcommerce_task2_df = (qcommerce_clean_df.withColumn("Effective_Order_Value",col("Order_Value") * (1 - col("Discount_Applied")))
    .withColumn("Price_Bucket",when(col("Effective_Order_Value") < 200, "LOW")
        .when((col("Effective_Order_Value") >= 200) & (col("Effective_Order_Value") <= 500),"MEDIUM")
        .otherwise("HIGH"))
    .groupBy("Price_Bucket").agg(
        count("*").alias("Count_Price_Bucket"),
        avg("Effective_Order_Value").alias("Avg_Effective_Value"))
    .orderBy(col("Avg_Effective_Value").desc()))

qcommerce_task2_df.show()

[Stage 27:=============================>                            (1 + 1) / 2]

+------------+------------------+-------------------+
|Price_Bucket|Count_Price_Bucket|Avg_Effective_Value|
+------------+------------------+-------------------+
|        HIGH|            268953|  740.4337238893675|
|      MEDIUM|            210429|  358.0973233400432|
|         LOW|            520618| 21.591204157544553|
+------------+------------------+-------------------+



In [30]:
qcommerce_task3_df = qcommerce_clean_df.withColumn("Age_Group", when(col("Customer_Age") < 25, "YOUNG") \
                                                   .when((col("Customer_Age") >= 25) & (col("Customer_Age") < 44), "ADULT") \
                                                   .otherwise("SENIOR")) \
                                                   .filter((col("Customer_Age") > 0) & (col("Customer_Age") <= 100)) \
                                                   .groupBy(col("Age_Group"), col("Product_Category")).agg(
                                                       count("*").alias("count_age_group"),
                                                       avg("Order_Value").alias("avg_order_value")
                                                   ).orderBy(col("Age_Group"), ascending=False)
qcommerce_task3_df.show()

[Stage 24:=============================>                            (1 + 1) / 2]

+---------+-------------------+---------------+-----------------+
|Age_Group|   Product_Category|count_age_group|  avg_order_value|
+---------+-------------------+---------------+-----------------+
|    YOUNG|      Personal Care|          23687| 568.364765176416|
|    YOUNG|             Snacks|          23774|571.4999866301071|
|    YOUNG|              Dairy|          24235|572.3825174487889|
|    YOUNG|          Beverages|          23885|571.5616625072918|
|    YOUNG|          Household|          23942|577.2101138994992|
|    YOUNG|          Groceries|          23881|571.6804158469649|
|    YOUNG|Fruits & Vegetables|          23660|570.9057398116498|
|   SENIOR|      Personal Care|          54025|570.9699125884629|
|   SENIOR|             Snacks|          54339|572.3606182995497|
|   SENIOR|          Groceries|          54353|571.9842917366681|
|   SENIOR|          Household|          54235|571.3643088361069|
|   SENIOR|Fruits & Vegetables|          53988|573.6265609516801|
|   SENIOR

In [39]:
#su._spark.stop()